# Lab 3: Image generation

Generate a PNG, compare image sizes and seeds, and handle the service's rate limit.

**Prerequisites:** [setup](../docs/00_prework.md) and a team key. Allow about 15 minutes, plus any queue or rate-limit delay. This lab saves images locally and does not provision cloud resources.

[Notebook index](README.md) · [API reference](../docs/inference.md) · [Troubleshooting](../docs/troubleshooting.md#inference)

These examples use the workshop service configuration. Confirm aliases and quotas with the organizers. Event details used as sample data are fictional.


## Step 1: Set up your environment

In [ ]:
# Install the packages used in this notebook.
%pip install -q "openai>=3,<4" python-dotenv pillow

In [ ]:
import os
from pathlib import Path
from dotenv import load_dotenv
from openai import OpenAI

# Run Jupyter from the workshop directory or its notebooks directory.
workshop_dir = Path.cwd().resolve()
if workshop_dir.name == "notebooks":
    workshop_dir = workshop_dir.parent
env_file = workshop_dir / ".env"
if not (workshop_dir / ".env.example").is_file() or not env_file.is_file():
    raise RuntimeError("Open Jupyter from the workshop directory after completing docs/00_prework.md.")
load_dotenv(env_file, override=True)

base_url = os.environ.get("OPENAI_BASE_URL", "").strip().rstrip("/")
api_key = os.environ.get("OPENAI_API_KEY", "").strip()
if not base_url or not api_key or api_key == "hk_paste_your_team_key_here":
    raise RuntimeError("Set OPENAI_BASE_URL and your team key in the workshop .env file.")
print("Endpoint:", base_url)
print("Team key loaded. Key value is hidden.")
client = OpenAI(base_url=base_url, api_key=api_key, timeout=60.0)


## Request reference

`POST /v1/images/generations` accepts `model="hackathon-image"`, a prompt, and a supported square size. Request one image with `response_format="b64_json"`. Decode `data[0].b64_json` to obtain the PNG.

The supplied sizes are `512x512`, `768x768`, and `1024x1024`. See the [API reference](../docs/inference.md) for other deployment limits.

In [ ]:
import base64

response = client.images.generate(
    model="hackathon-image",
    prompt="A yellow bicycle, simple icon",
    size="512x512",
    response_format="b64_json",
)
png_bytes = base64.b64decode(response.data[0].b64_json)
open("minimal.png", "wb").write(png_bytes)
print("Saved minimal.png,", len(png_bytes), "bytes")

## Step 2: Generate and inspect an image

Decode the response and open it with Pillow. Inspect the pixel dimensions before saving or sending it to a frontend.

In [ ]:
import base64
import io
import time
from IPython.display import display
from PIL import Image

def to_image(response):
    """Decode the first image in an images.generate response."""
    return Image.open(io.BytesIO(base64.b64decode(response.data[0].b64_json)))

start = time.time()
response = client.images.generate(
    model="hackathon-image",
    prompt="A friendly robot holding a coffee cup, flat vector illustration, warm colors",
    size="512x512",
    n=1,
    response_format="b64_json",
)
print(f"Generated in {time.time() - start:.1f}s | fields: {list(response.model_dump().keys())}")

image = to_image(response)
print("Size:", image.size, "| mode:", image.mode)
image.save("lab3-first.png")
display(image)

## Step 3: Compare sizes

Generate at the two larger supported sizes. Record elapsed time and inspect the results. The queue is shared, so a single timing does not establish a reliable performance comparison.

In [ ]:
prompt = "A lighthouse on a rocky coast at dusk, oil painting"

for size in ("768x768", "1024x1024"):
    start = time.time()
    r = client.images.generate(model="hackathon-image", prompt=prompt, size=size, response_format="b64_json")
    img = to_image(r)
    print(f"{size}: {time.time() - start:.1f}s, {img.size}")
    img.save(f"lab3-{size}.png")

display(img)

## Step 4: Inspect a rejected option

The supplied deployment supports base64 responses rather than hosted image URLs. The next cell intentionally requests `response_format="url"` to inspect the validation response. Avoid unsupported requests in your application.

In [ ]:
import openai

try:
    client.images.generate(model="hackathon-image", prompt="a red cube", size="512x512", response_format="url")
except openai.APIStatusError as e:
    print("Status:", e.status_code)
    print("Body:  ", e.response.text[:160])

## Step 5: Compare seeds

Pass `seed` through `extra_body` to compare runs under the same model and deployment configuration. The supplied service accepts integers from 0 to 2,147,483,647. Repeatability can change with the underlying model or runtime.

In [ ]:
prompt = "A small wooden cabin in a snowy forest, morning light, watercolor"
images = []
for seed in (7, 42):
    r = client.images.generate(model="hackathon-image", prompt=prompt, size="512x512",
                               response_format="b64_json", extra_body={"seed": seed})
    images.append(to_image(r))

side_by_side = Image.new("RGB", (1024 + 16, 512), "white")
side_by_side.paste(images[0], (0, 0))
side_by_side.paste(images[1], (512 + 16, 0))
display(side_by_side)

## Step 6: Handle quota and queue limits

The service may return `429` for a team quota or a busy image queue. The helper uses `Retry-After` or the service-specific `x-ratelimit-reset` delay, adds a small random delay, and bounds the retry attempts.

Several image requests have already been made in this notebook. Their timing determines whether this next call encounters a limit.

In [ ]:
import random

quota_client = OpenAI(base_url=base_url, api_key=api_key, timeout=60.0, max_retries=0)  # Retries are handled here so you can see them.

def generate_with_retry(prompt, size="512x512", seed=0, attempts=4):
    for attempt in range(attempts):
        try:
            return quota_client.images.generate(
                model="hackathon-image", prompt=prompt, size=size,
                response_format="b64_json", extra_body={"seed": seed},
            )
        except openai.RateLimitError as e:
            if attempt == attempts - 1:
                raise
            headers = e.response.headers
            wait = float(headers.get("retry-after") or headers.get("x-ratelimit-reset") or 10)
            wait += random.uniform(0, 2)
            print(f"429 received (limit {headers.get('x-ratelimit-limit', '?')}/min); waiting {wait:.0f}s")
            time.sleep(wait)
    raise RuntimeError("Gave up after repeated 429 responses")

r = generate_with_retry("A paper airplane over a city skyline, minimal poster")
print("Succeeded.")
display(to_image(r))

A successful first attempt is normal. The retry path is used only when the service returns `429`; this notebook does not deliberately flood the service to trigger it.

## Use this in an application

Keep generation on the backend so the key stays private. Return the generated file or store it in [Object Storage](../docs/01_knowledge.md#4-use-the-bucket-from-your-own-app). This deployment supports text-to-image requests, not image editing or image input to chat.

Measure response time under your expected workload and handle queue delays in the interface.

## Validation

In [ ]:
raw = base64.b64decode(response.data[0].b64_json)
assert raw[:8] == b"\x89PNG\r\n\x1a\n", "not a PNG"
assert image.size == (512, 512)
print("Lab 3 complete.")

## Clean up

The following cell removes only the filenames used by this notebook from the kernel's working directory. Save any images you want to keep first.

In [ ]:
from pathlib import Path

for filename in ("minimal.png", "lab3-first.png", "lab3-768x768.png", "lab3-1024x1024.png"):
    path = Path(filename)
    if path.exists():
        path.unlink()
        print("Removed", path)


## Continue with your project

Use the [Object Storage example](../docs/01_knowledge.md#4-use-the-bucket-from-your-own-app) to upload an image and retrieve it. Choose [Linode](../docs/03_body.md) or [LKE](../docs/04_lke.md) if you need a hosted backend. Return to the [notebook index](README.md) for other capabilities.